
# Minimum radiation field U_min: DL07 and THEMIS agree on the FIR peak

The starlight intensity floor U_min sets the temperature of the diffuse-ISM
component in template-based dust libraries. We compare the Draine & Li
2007 grid (fixed q_PAH = 2.5%) and the THEMIS grid (fixed q_HAC = 0.17) at
three matched U_min values to highlight that the FIR-peak position is
remarkably consistent between the two grain-physics paradigms, while THEMIS
predicts a stronger mid-IR continuum from its hydrogenated amorphous carbon
component.

References:
    Draine, B.T. & Li, A. 2007, ApJ, 657, 810.
    Jones, A.P. et al. 2013/2017 — THEMIS model series.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np

from tengri import data_path
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

SHOWN_UMIN = (0.5, 2.0, 10.0)
C_AA_PER_S = 2.99792458e18

with h5py.File(data_path("dl07_templates.h5"), "r") as f:
    wave_aa_dl = np.asarray(f["wavelength"][:])
    umin_dl = np.asarray(f["umin_grid"][:])
    qpah_dl = np.asarray(f["qpah_grid"][:])
    single_u_dl = np.asarray(f["single_u"][:])
i_qpah = int(np.argmin(np.abs(qpah_dl - 2.5)))
nu_dl = C_AA_PER_S / wave_aa_dl
wave_um_dl = wave_aa_dl * 1.0e-4

with h5py.File(data_path("themis_templates.h5"), "r") as f:
    wave_aa_th = np.asarray(f["wavelength_aa"][:])
    umin_th = np.asarray(f["umin_grid"][:])
    qhac_th = np.asarray(f["qhac_grid"][:])
    single_u_th = np.asarray(f["single_u"][:])
i_qhac = int(np.argmin(np.abs(qhac_th - 0.17)))
nu_th = C_AA_PER_S / wave_aa_th
wave_um_th = wave_aa_th * 1.0e-4

fig, ax = plt.subplots(figsize=(7.5, 4.6))
cmap = plt.get_cmap("viridis")

for k, target in enumerate(SHOWN_UMIN):
    color = cmap(k / (len(SHOWN_UMIN) - 1))
    i_dl = int(np.argmin(np.abs(umin_dl - target)))
    ax.plot(
        wave_um_dl,
        nu_dl * single_u_dl[i_qpah, i_dl],
        color=color,
        lw=1.4,
        ls="-",
        label=rf"DL07, $U_{{\rm min}}={umin_dl[i_dl]:.2f}$",
    )
    i_th = int(np.argmin(np.abs(umin_th - target)))
    ax.plot(
        wave_um_th,
        nu_th * single_u_th[i_qhac, i_th],
        color=color,
        lw=1.4,
        ls="--",
        label=rf"THEMIS, $U_{{\rm min}}={umin_th[i_th]:.2f}$",
    )

ax.set(
    xscale="log",
    yscale="log",
    xlim=(2.0, 1.0e3),
    xlabel=r"$\lambda\ [\mu\mathrm{m}]$",
    ylabel=r"$\nu L_\nu$  [template units]",
)
ax.legend(loc="lower right", frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
plt.savefig("plot_umin_cross_library.png", dpi=150, bbox_inches="tight")